# InsightFlow Executive — EDA + Preprocessing + Fine-tuning

**Pipeline complet : analyse → nettoyage → équilibrage → fine-tuning**

| Étape | Description |
|-------|-------------|
| 1. Setup | GPU + dépendances + clone repo |
| 2. EDA | Analyse exploratoire du dataset |
| 3. Preprocessing | Nettoyage + déduplication + équilibrage + split 70/15/15 |
| 4. Fine-tuning sentiment | XLM-RoBERTa — POSITIVE/NEUTRAL/NEGATIVE |
| 5. Fine-tuning émotion | XLM-RoBERTa multilingue FR+EN — 5 classes business |
| 6. Résultats + download | Comparaison + téléchargement modèles |

> **⚠ Important** : Runtime → Change runtime type → **T4 GPU** avant de commencer

In [ ]:
# ── Étape 1a : Vérifier le GPU ────────────────────────────────
import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠ Pas de GPU — Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Étape 1b : Installer les dépendances ─────────────────────
!pip install -q transformers datasets torch scikit-learn accelerate evaluate matplotlib seaborn tabulate

In [ ]:
# ── Étape 1c : Cloner le repo GitHub ─────────────────────────
!git clone https://github.com/sarrahafsi/InsightFlowExecutive.git
%cd InsightFlowExecutive
!ls ml/dataset/

In [ ]:
# ── Étape 1c (alternative) : Upload manuel si repo privé ─────
# Décommente et exécute si le clone échoue

# from google.colab import files
# import os
# os.makedirs('InsightFlowExecutive/ml/dataset', exist_ok=True)
# os.makedirs('InsightFlowExecutive/ml/models', exist_ok=True)
# print('Upload insightflow_synthetic_full.csv')
# uploaded = files.upload()
# for fname in uploaded:
#     os.rename(fname, f'InsightFlowExecutive/ml/dataset/{fname}')
# for script in ['eda.py', 'preprocess.py', 'finetune.py']:
#     print(f'Upload {script}')
#     files.upload()
#     os.rename(script, f'InsightFlowExecutive/ml/{script}')
# %cd InsightFlowExecutive

---
## Étape 2 — EDA (Exploratory Data Analysis)

In [ ]:
# ── EDA complète sur le dataset full ─────────────────────────
!python ml/eda.py --lang full

In [ ]:
# ── Afficher les charts EDA ───────────────────────────────────
from IPython.display import Image, display
import os

charts = [
    ('ml/charts/eda/0_eda_dashboard.png',              'Dashboard EDA complet'),
    ('ml/charts/eda/1_class_distributions.png',        'Distribution des classes'),
    ('ml/charts/eda/2_text_lengths.png',               'Longueur des textes'),
    ('ml/charts/eda/3_sentiment_emotion_correlation.png','Corrélation Sentiment × Émotion'),
    ('ml/charts/eda/4_business_distribution.png',      'Business labels'),
]
for path, title in charts:
    if os.path.exists(path):
        print(f'\n── {title} ──')
        display(Image(path))

---
## Étape 3 — Preprocessing (nettoyage + équilibrage + split)

In [ ]:
# ── Preprocessing complet ─────────────────────────────────────
# strategy : oversample (défaut) | undersample | none
!python ml/preprocess.py --lang full --strategy oversample

In [ ]:
# ── Vérifier les fichiers nettoyés ───────────────────────────
import csv, json, os

for fname in [
    'clean_sentiment_train.csv', 'clean_sentiment_val.csv', 'clean_sentiment_test.csv',
    'clean_emotion_train.csv',   'clean_emotion_val.csv',   'clean_emotion_test.csv',
]:
    path = f'ml/dataset/{fname}'
    if os.path.exists(path):
        with open(path) as f:
            n = sum(1 for _ in f) - 1
        print(f'✓ {fname:<45} {n:>5} lignes')

# Afficher le rapport
report_path = 'ml/dataset/preprocessing_report.json'
if os.path.exists(report_path):
    report = json.load(open(report_path))
    print(f'\nRapport preprocessing :')
    print(f'  Stratégie   : {report["strategy"]}')
    print(f'  Doublons supprimés : {report["duplicates_removed"]}')
    print(f'  Sentiment → train:{report["sentiment"]["train"]} | val:{report["sentiment"]["val"]} | test:{report["sentiment"]["test"]}')
    print(f'  Émotion   → train:{report["emotion"]["train"]} | val:{report["emotion"]["val"]} | test:{report["emotion"]["test"]}')

---
## Étape 4 — Fine-tuning Sentiment (XLM-RoBERTa)
**Durée estimée : ~25 min sur T4**

In [ ]:
!python ml/finetune.py --task sentiment --model xlm --lang full

---
## Étape 5 — Fine-tuning Émotion (XLM-RoBERTa multilingue FR+EN — 5 classes business)
**Durée estimée : ~15 min sur T4**

In [ ]:
!python ml/finetune.py --task emotion --lang full

---
## Étape 6 — Résultats comparatifs

In [ ]:
import json, os

print('=' * 65)
print('  InsightFlow — Fine-tuning Results')
print('=' * 65)

models = [
    ('insightflow-sentiment-xlm-v1',  'Sentiment — XLM-RoBERTa (FR+EN)'),
    ('insightflow-emotion-xlm-v1',    'Émotion   — XLM-RoBERTa multilingue (5 classes)'),
]

for model_dir, label in models:
    path = f'ml/models/{model_dir}/training_summary.json'
    if os.path.exists(path):
        s = json.load(open(path))
        print(f'\n  {label}')
        print(f'  Dataset  : {s["dataset_size"]} samples | Lang : {s["language"]}')
        print(f'  Accuracy : {s["eval_accuracy"]*100:.2f}%')
        print(f'  F1-macro : {s["eval_f1_macro"]*100:.2f}%')
    else:
        print(f'\n  {label} — pas encore entraîné')

print('\n' + '=' * 65)

In [ ]:
# ── Télécharger le modèle émotion fine-tuné ──────────────────
import shutil, os
from google.colab import files

model_dir = 'insightflow-emotion-xlm-v1'
path = f'ml/models/{model_dir}'
if os.path.exists(path):
    zip_path = f'{model_dir}.zip'
    shutil.make_archive(model_dir, 'zip', 'ml/models', model_dir)
    files.download(zip_path)
    print(f'✓ Téléchargé : {zip_path}')
else:
    print(f'✗ Modèle non trouvé : {model_dir}')